# Overlaid Graphs

In this chapter, we have learned how to visualize data by drawing graphs. A common use of such visualizations is to compare two datasets. In this section, we will see how to *overlay* plots, that is, draw them in a single graphic on a common pair of axes.

For the overlay to make sense, the graphs that are being overlaid must represent the same variables and be measured in the same units. 

To draw overlaid graphs, `hvplot.scatter`, `hvplot.line`, and `hvplot.barh` can all be called in the same way. For scatter and line plots, one column must serve as the common horizontal axis for all the overlaid graphs. For `barh`, one column must serve as the common axis which is the set of categories. The general call looks like:

`name_of_dataframe.hvplot.method(x=column_label_of_common_axis, y=list_of_labels_of_variables_to_plot)`

More commonly, we will first select only the columns needed for our graph. hvPlot still names the columns to draw. A single axis argument does not mean "every other column."


In [1]:
import polars as pl
import hvplot.polars
path_data = '../../../assets/data/'


## Overlaid Scatter Plots

The DataFrame `sons_heights` is part of a historical data set on the heights of parents and their children. Specifically, the population consists of 179 men who were the first-born in their families. The data are their own heights and the heights of their parents. All heights were measured in inches.

In [2]:
sons_heights = pl.read_csv(path_data + 'sons_heights.csv')
sons_heights


father,mother,son
f64,f64,f64
78.5,67.0,73.2
75.5,66.5,73.5
75.0,64.0,71.0
75.0,64.0,70.5
75.0,58.5,72.0
…,…,…
64.0,64.0,70.5
64.0,63.0,64.5
64.0,60.0,66.0


`hvplot.scatter` visualizes how the sons' heights are related to the heights of both their parents. In the graph, the sons' heights will form the common horizontal axis. 

In [3]:
# POLARS_CHANGE: one column name used to draw every other column. hvplot.scatter takes the vertical columns in y.
# Blue is the father and gold is the mother, matching the sentences below.
sons_heights.hvplot.scatter(
    x='son',
    y=['father', 'mother'],
    alpha=0.5,
    color=['#002fa7', '#706700'],
)


A scatter plot shows the relationship between the height of a 'son' on the x-axis and the heights of parents on the y-axis. Dark blue dots represent the 'father' data, which generally cluster higher on the y-axis, while gold dots represent the 'mother' data, clustering noticeably lower. Both sets of data show a slight upward trend, indicating that taller sons tend to have taller parents.

The call names the sons' heights on the common horizontal axis and lists `father` and `mother` as the vertical columns. Python drew two scatter plots: one each for the relation between the son's height and those two.

Each point represents a row of the DataFrame, that is, a "father, mother, son" trio. For all points, the horizontal axis represents the son's height. In the blue points, the vertical axis represents the father's height. In the gold points, the vertical axis represents the mother's heights.

Both the gold and the blue scatter plots slope upwards and show a positive association between the sons' heights and the heights of both their parents. The blue (fathers) plot is in general higher than the gold, because the fathers were in general taller than the mothers.

## Overlaid Line Plots

Our next example involves data on children of more recent times. We will return to the Census data DataFrame `us_pop`, created below again for reference. From this DataFrame, we will extract the counts of all children in each of the age categories 0 through 18 years.

In [4]:
# Read the full Census DataFrame
data = 'http://www2.census.gov/programs-surveys/popest/technical-documentation/file-layouts/2010-2019/nc-est2019-agesex-res.csv'
full_census_table = pl.read_csv(data)

# Select columns from the full DataFrame and rename some of them
partial_census_table = full_census_table.select('SEX', 'AGE', 'POPESTIMATE2014', 'POPESTIMATE2019')
us_pop = partial_census_table.rename({'POPESTIMATE2014': '2014', 'POPESTIMATE2019': '2019'})

# Access the rows corresponding to all children, ages 0-18
children = us_pop.filter((pl.col('SEX') == 0) & (pl.col('AGE') < 19)).drop('SEX')
children.show(None)


AGE,2014,2019
i64,i64,i64
0,3954787,3783052
1,3948891,3829599
2,3958711,3922044
3,4005928,3998665
4,4004032,4043323
5,4004576,4028281
6,4133372,4017227
7,4152666,4022319
8,4118349,4066194


We can now draw two overlaid line plots, showing the numbers of children in the different age groups for each of the years 2014 and 2019. The call is analogous to the `hvplot.scatter` call in the previous example.

In [5]:
# POLARS_CHANGE: plot('AGE') drew every other column against AGE. 2014 is blue and 2019 is gold.
children.hvplot.line(x='AGE', y=['2014', '2019'], color=['#002fa7', '#706700'])


A line graph compares population data by 'AGE' on the x-axis, ranging from 0 to 18, against a numerical scale in millions on the y-axis. Two lines track the data: a dark blue line for '2014' and a gold line for '2019'. Both lines follow a fluctuating but generally upward trend as age increases, with '2019' starting significantly lower at age 0 but rising to cross over and match or exceed '2014' values at multiple points.

Though the horizontal axis labels include some half-integers, it's important to remember that we only have data at ages 0, 1, 2, and so on. The line plots "join the dots" in between.

The two graphs cross each other in a few places. For example, there were more 6-year-olds in 2014 than in 2019, and there were more 12-year-olds in 2019 than in 2014.

Of course, the 12-year-olds in 2019 mostly consist of the children who were 7-year-olds in 2014. To see this on the plots, compare the gold graph at `AGE` 12 and the blue graph at `AGE` 7. You will notice that the gold graph (2019) looks very much like the blue graph (2014) slid over to the right by 5 years. The slide is accompanied by a slight rise due to the net effect of children who entered the country between 2014 and 2019 outnumbering those who left. Fortunately at these ages there is not much loss of life.

## Bar Charts

The Kaiser Family Foundation has complied Census data on the distribution of race and ethnicity in the U.S. The Foundation's website provides compilations of data for [the entire U.S. population](http://kff.org/other/state-indicator/distribution-by-raceethnicity/) in 2019, as well as for [U.S. children](http://kff.org/other/state-indicator/children-by-raceethnicity/) who were younger than 18 years old that year.

The DataFrame `usa_ca` is adapted from their data for the United States and California. The columns represent everyone in the U.S.A., everyone in California, children in the U.S.A., and children in California. 

The body of the DataFrame contains percents in the different categories. Each column shows the distribution of the `Ethnicity/Race` variable in the group of people corresponding to that column. So in each column, the entries add up to 100. The `API` category consists of Asians and Pacific Islanders including Native Hawaiians. The `Other` category includes Native Americans, Alaskan natives, and people who identify with multiple races.

In [6]:
usa_ca = pl.read_csv(path_data + 'usa_ca_2019.csv')
usa_ca


Ethnicity/Race,USA All,CA All,USA Children,CA Children
str,f64,f64,f64,f64
"""API""",5.8,15.1,4.9,11.5
"""Black""",12.2,5.3,13.4,4.9
"""Hispanic""",18.5,39.5,25.6,52.1
"""White""",60.1,36.4,50.0,25.5
"""Other""",3.4,3.7,6.1,6.0


It is natural to want to compare these distributions. It makes sense to compare the columns directly, because all the entries are percents and are therefore on the same scale.

`hvplot.barh` visualizes the comparisons by drawing multiple bar charts on the same axes. The call is analogous to those for `hvplot.scatter` and `hvplot.line`: we have to specify the common axis of categories. 

In [7]:
usa_ca.hvplot.barh(
    'Ethnicity/Race',
    ['USA All', 'CA All', 'USA Children', 'CA Children'],
    color=['#002fa7', '#706700', '#006a94', '#1e6400'],
)


A horizontal bar chart compares demographic percentages across different groups labeled 'Ethnicity/Race' on the y-axis, including API, Black, Hispanic, White, and Other. Each demographic contains four bars representing different categories: 'USA All' (dark blue), 'CA All' (gold), 'USA Children' (light blue), and 'CA Children' (green). The data shows distinct regional and age variations, but generally within an Ethnicity/Race, the bars are either all on the shorter end, like for API, Black, and Other, or all on the longer end, like for Hispanic and White.

While drawing the overlaid bar charts is straightforward, there is a bit too much information on this graph for us to be able to sort out similarities and differences between populations. It is much easier to compare the populations one pair at a time. 

Let's start by comparing the entire populations of the U.S.A. and California. 

In [8]:
usa_ca.select('Ethnicity/Race', 'USA All', 'CA All').hvplot.barh(
    'Ethnicity/Race',
    ['USA All', 'CA All'],
    color=['#002fa7', '#706700'],
)


A horizontal bar chart compares demographic percentages across different groups labeled 'Ethnicity/Race' on the y-axis: API, Black, Hispanic, White, and Other. Each demographic contains two bars representing different categories: 'USA All' (dark blue) and 'CA All' (gold). The longest bar is for White-USA All at 60. The next longest bar is Hispanic-CA All at just below 40 and White-CA All between 30 and 40. The remaining bars are all less than 20 long.

The two distributions are quite different. California has higher percents in the `API` and `Hispanic` categories, and correspondingly lower percents in the`Black` and `White` categories. The percents in the `Other` category are quite similar in the two populations. The differences are largely due to California's geographical location and patterns of immigration and migration, both historically and in more recent decades. 

As you can see from the graph, almost 40% of the Californian population in 2019 was `Hispanic`. A comparison with the population of children in the state indicates that the `Hispanic` proportion is likely to be greater in future years. Among Californian children in 2019, more than 50% were in the `Hispanic` category.

In [9]:
usa_ca.select('Ethnicity/Race', 'CA All', 'CA Children').hvplot.barh(
    'Ethnicity/Race',
    ['CA All', 'CA Children'],
    color=['#002fa7', '#706700'],
)


A horizontal bar chart compares demographic percentages across different groups labeled 'Ethnicity/Race' on the y-axis: API, Black, Hispanic, White, and Other. Each demographic contains two bars representing different categories: 'CA All' (dark blue) and 'CA Children' (gold). The longest bar is just above 50 for Hispanic-CA Children, followed by Hispanic-CA All around 40, and White-CA All below that. The remain bars have lengths below 30.

More complex data sets naturally give rise to varied and interesting visualizations, including overlaid graphs of different kinds. To analyze such data, it helps to have some more skills in data manipulation, so that we can get the data into a form that allows us to use methods like those in this section. In the next chapter we will develop some of these skills.